In [1]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit

In [2]:
# let's load some data
db = SQLDatabase.from_uri("sqlite:///../data/Chinook.db")

In [3]:
db.run("SELECT * FROM Artist LIMIT 5;")

"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]"

In [4]:
# this SQLDatabase object has some nice methods to work with the database
# they will be used by our agent
db.get_usable_table_names()

['Album',
 'Artist',
 'Customer',
 'Employee',
 'Genre',
 'Invoice',
 'InvoiceLine',
 'MediaType',
 'Playlist',
 'PlaylistTrack',
 'Track']

In [5]:
# let's init our LLM
model = init_chat_model("openai:gpt-4o-mini")

In [6]:

# with langchain is super easy to create a SQL toolkit to interact with sql data
toolkit = SQLDatabaseToolkit(db=db, llm=model)

tools = toolkit.get_tools()

for tool in tools:
    print(f"{tool.name}: {tool.description}\n")

sql_db_query: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables: Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



In [7]:
# let's now create a nice agent

system_prompt = """
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
""".format(
    dialect=db.dialect,
    top_k=5,
)

agent = create_agent(
    model,
    tools,
    system_prompt=system_prompt,
)

In [8]:
question = "Which genre on average has the longest tracks?"

responses = agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
)

responses["messages"][-1].pretty_print()

# You can also get the response in streaming mode
#
# for step in agent.stream(
#     {"messages": [{"role": "user", "content": question}]},
#     stream_mode="values",
# ):
#     step["messages"][-1].pretty_print()

================================== Ai Message ==================================

The genre with the longest average track duration is **"Sci Fi & Fantasy"** with an average track length of approximately **2911783 milliseconds** (or about 48 minutes and 31 seconds). Here are a few more genres and their average track durations:

1. **"Science Fiction"** - 2625549 milliseconds (about 43 minutes and 45 seconds)
2. **"Drama"** - 2575283 milliseconds (about 42 minutes and 55 seconds)
3. **"TV Shows"** - 2145041 milliseconds (about 35 minutes and 45 seconds)
4. **"Comedy"** - 1585263 milliseconds (about 26 minutes and 25 seconds)


In [9]:
import mlflow

mlflow.langchain.autolog()

In [10]:
mlflow.set_experiment("langchain-sqlite-agent")

2025/10/21 14:59:05 INFO mlflow.tracking.fluent: Experiment with name 'langchain-sqlite-agent' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///Users/vitale/Projects/fun-with-mlflow/notebooks/mlruns/166447074697988074', creation_time=1761051545262, experiment_id='166447074697988074', last_update_time=1761051545262, lifecycle_stage='active', name='langchain-sqlite-agent', tags={}>

In [11]:

question = "List the 5 artists with the most albums."

# give the reponse directly
responses = agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
)

responses["messages"][-1].pretty_print()

================================== Ai Message ==================================

The 5 artists with the most albums are:

1. Iron Maiden - 21 albums
2. Led Zeppelin - 14 albums
3. Deep Purple - 11 albums
4. U2 - 10 albums
5. Metallica - 10 albums
